# 08 — Phân tích lỗi SVM bằng ảnh thật

Phân tích bao phủ toàn bộ dự đoán test đã lưu. SVM đa lớp trả `decision_function` dạng điểm quyết định một-vs-phần-còn-lại từ các phân loại cặp lớp. Điểm và khoảng cách giữa hai điểm cao nhất không phải xác suất hay độ tin cậy đã hiệu chỉnh.


In [ ]:
# KAGGLE_INPUT_SETUP: tìm code và DATA trong các Dataset đã gắn.
from pathlib import Path
import os

kaggle_input = Path('/kaggle/input')
dataset_roots = []
if kaggle_input.is_dir():
    dataset_roots.extend(kaggle_input.glob('datasets/*/*'))
    dataset_roots.extend(kaggle_input.glob('*'))
dataset_roots = [p for p in dataset_roots if p.is_dir()]

project_hint = os.environ.get('CS231_PROJECT_ROOT')
project_candidates = ([Path(project_hint)] if project_hint else
                      [Path.cwd(), *Path.cwd().parents, *dataset_roots])
project_matches = sorted({p.resolve() for p in project_candidates
                          if (p / 'src/SVM/pipeline.py').is_file()})
if len(project_matches) != 1:
    raise FileNotFoundError(
        'Cần gắn đúng Dataset code chứa src/SVM/pipeline.py; '
        'nếu có nhiều bản, đặt CS231_PROJECT_ROOT. '
        f'Tìm thấy: {project_matches}'
    )
project = project_matches[0]

data_hint = os.environ.get('CS231_DATA_ROOT')
data_candidates = ([Path(data_hint)] if data_hint else
                   [project / 'DATA', *(p / 'DATA' for p in dataset_roots)])
data_matches = sorted({p.resolve() for p in data_candidates
                       if (p / 'train/10000').is_dir()})
if len(data_matches) != 1:
    raise FileNotFoundError(
        'Cần gắn đúng Dataset DATA chứa DATA/train/10000; '
        'nếu có nhiều bản, đặt CS231_DATA_ROOT. '
        f'Tìm thấy: {data_matches}'
    )
data = data_matches[0]

os.environ['CS231_PROJECT_ROOT'] = str(project)
os.environ['CS231_DATA_ROOT'] = str(data)
print('Code:', project, (project / 'src/SVM/pipeline.py').is_file())
print('DATA:', data, (data / 'train/10000').is_dir())


In [ ]:
PREVIOUS_ARTIFACT = 'test_final/metrics_test.json'
# PREVIOUS_NOTEBOOK_OUTPUT: khôi phục toàn bộ kết quả của notebook ngay trước đó.
from pathlib import Path
from shutil import copytree
import os

if Path('/kaggle/working').is_dir():
    notebook_root = Path('/kaggle/input/notebooks')
    attached = sorted(p for p in notebook_root.glob('*/*') if p.is_dir())
    if len(attached) != 1:
        raise FileNotFoundError(
            'Hãy gắn đúng một Notebook Output của bước ngay trước đó. '
            f'Đang thấy: {attached}'
        )
    configs = [p for p in attached[0].rglob('experiment_config.json')
               if p.parent.name == 'inputs' and p.parent.parent.name == 'results_224']
    if len(configs) != 1:
        raise FileNotFoundError(
            'Notebook Output đã gắn phải chứa results_224/inputs/experiment_config.json. '
            f'Tìm thấy: {configs}'
        )
    source = configs[0].parents[1]
    expected = source / PREVIOUS_ARTIFACT
    if not expected.is_file():
        raise FileNotFoundError(
            f'Output không phải bản chạy thành công của bước ngay trước: thiếu {expected}'
        )
    target = Path(os.environ.get('CS231_SVM_RESULTS_ROOT',
                                 '/kaggle/working/CS231_SVM/results_224'))
    copytree(source, target, dirs_exist_ok=True)
    for relative in ('inputs/train_manifest.csv', 'inputs/dev_manifest.csv'):
        if not (target / relative).is_file():
            raise FileNotFoundError(f'Output thiếu {target / relative}')
    print('Đã chép:', source, '→', target)
else:
    print('Chạy local: dùng results_224 hiện có trong repository.')


In [ ]:
from pathlib import Path
import os, sys

# Có thể chỉ rõ ba đường dẫn bằng biến môi trường khi gắn repository/dataset trên Kaggle.
root_hint = os.environ.get('CS231_PROJECT_ROOT')
root_candidates = [Path(root_hint)] if root_hint else [Path.cwd(), *Path.cwd().parents]
if Path('/kaggle/input').is_dir():
    root_candidates += list(Path('/kaggle/input').glob('*'))
    root_candidates += list(Path('/kaggle/input').glob('*/*'))
PROJECT_ROOT = next((p.resolve() for p in root_candidates
                     if (p / 'src/SVM/pipeline.py').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Không tìm thấy repository. Đặt CS231_PROJECT_ROOT.')
sys.path.insert(0, str(PROJECT_ROOT))

data_hint = os.environ.get('CS231_DATA_ROOT')
data_candidates = [Path(data_hint)] if data_hint else [PROJECT_ROOT / 'DATA']
if not data_hint and Path('/kaggle/input').is_dir():
    data_candidates += list(Path('/kaggle/input').glob('*/DATA'))
    data_candidates += list(Path('/kaggle/input').glob('*/*/DATA'))
DATA_ROOT = next((p.resolve() for p in data_candidates
                  if (p / 'train/10000').is_dir()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Không tìm thấy DATA/train/10000. Đặt CS231_DATA_ROOT.')

result_hint = os.environ.get('CS231_SVM_RESULTS_ROOT')
if result_hint:
    RESULTS_ROOT = Path(result_hint).resolve()
elif Path('/kaggle/working').is_dir():
    RESULTS_ROOT = Path('/kaggle/working/CS231_SVM/results_224')
else:
    RESULTS_ROOT = PROJECT_ROOT / 'src/SVM/results_224'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print('Repository:', PROJECT_ROOT)
print('DATA:', DATA_ROOT)
print('Kết quả:', RESULTS_ROOT)


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.SVM.pipeline import (
    CLASSES, FEATURE_FOLDERS, manifest_for_split, extract_matrix, l2,
    save_features, load_features, same_manifest, run_dev_grid, save_dev_result,
    save_json, hsv12, hog_raw,
)
from src.preprocessing import read_rgb224


## 1. Đối chiếu cấu hình, feature và dự đoán test


In [ ]:
from src.SVM.pipeline import sha256
from src.evaluation.value_metrics import evaluate_classification
from sklearn.svm import SVC
TEST = RESULTS_ROOT / 'test_final'
OUTPUT = RESULTS_ROOT / 'error_analysis'
OUTPUT.mkdir(parents=True, exist_ok=True)
lock_path = RESULTS_ROOT / 'selection/locked_svm_config.json'
lock = json.loads(lock_path.read_text(encoding='utf-8'))
if lock_path.read_bytes() != (TEST / 'locked_svm_config_used.json').read_bytes():
    raise ValueError('Khóa hiện tại khác khóa đã dùng trên test')
for key, expected in lock['sha256'].items():
    domain, relative = key.split('/', 1)
    root = RESULTS_ROOT if domain == 'results' else PROJECT_ROOT
    path = root / relative
    if not path.is_file() or sha256(path) != expected:
        raise ValueError('File đã khóa thay đổi: ' + key)
X_train, y_train, train_manifest = load_features(RESULTS_ROOT / lock['feature_folder'], 'train')
X_test = np.load(TEST / 'X_test.npy', mmap_mode='r')
test_manifest = pd.read_csv(TEST / 'test_manifest.csv')
predictions = pd.read_csv(TEST / 'predictions_test.csv')
if not predictions.relative_path.equals(test_manifest.relative_path):
    raise ValueError('Dự đoán và manifest test sai thứ tự')
y_test = predictions.y_true_vnd.to_numpy(dtype=np.int64)
y_pred = predictions.y_pred_vnd.to_numpy(dtype=np.int64)
if not np.array_equal(y_test, test_manifest.label_vnd):
    raise ValueError('Nhãn test không khớp manifest')
scores = evaluate_classification(y_test, y_pred)
if scores != json.loads((TEST / 'metrics_test.json').read_text(encoding='utf-8')):
    raise ValueError('Metric không khớp dự đoán đã lưu')
print('Đúng:', int((y_test == y_pred).sum()), 'Sai:', int((y_test != y_pred).sum()))


## 2. Lỗi theo lớp, cặp nhầm và chênh lệch VND

Một cặp xuất hiện nhiều nhất có thể khác cặp đóng góp tổng chênh lệch VND lớn nhất. Cả số lần và tổng chênh lệch đều cần được báo cáo.


In [ ]:
per_class = pd.DataFrame(scores['per_class'])
per_class['error_rate'] = per_class['fn'] / per_class['support']
cm = np.asarray(scores['confusion_matrix'])
pairs = []
for i, a in enumerate(CLASSES):
    for j in range(i + 1, len(CLASSES)):
        b = CLASSES[j]
        count = int(cm[i, j] + cm[j, i])
        pairs.append({'class_a': a, 'class_b': b, 'a_to_b': int(cm[i, j]),
                      'b_to_a': int(cm[j, i]), 'total_errors': count,
                      'gap_vnd': abs(a - b), 'total_gap_vnd': count * abs(a - b)})
pairs = pd.DataFrame(pairs).sort_values(
    ['total_errors', 'class_a', 'class_b'], ascending=[False, True, True]).reset_index(drop=True)
directed = pd.DataFrame(scores['error_pairs']).sort_values(
    ['total_gap_vnd', 'true_vnd', 'predicted_vnd'],
    ascending=[False, True, True]).reset_index(drop=True)
per_class.to_csv(OUTPUT / 'per_class_analysis.csv', index=False)
pairs.to_csv(OUTPUT / 'error_pairs_bidirectional.csv', index=False)
directed.to_csv(OUTPUT / 'error_pairs_cost.csv', index=False)
display(per_class[['denomination_vnd', 'support', 'fn', 'error_rate', 'f1']])
display(pairs.head(5))
display(directed.head(5))


## 3. Tính lại điểm quyết định cho toàn bộ test

`break_ties=True` khiến nhãn dự đoán nhất quán với điểm cao nhất. Khoảng cách giữa hai điểm cao nhất chỉ dùng để chọn ví dụ và mô tả quyết định trong mô hình hiện tại; không diễn giải là xác suất đúng.


In [ ]:
model = SVC(**lock['svm'])
model.fit(X_train, y_train)
if not np.array_equal(model.predict(X_test), y_pred):
    raise ValueError('Dự đoán tính lại khác CSV test')
decision = model.decision_function(X_test)
classes = model.classes_.astype(int)
if decision.shape != (len(y_test), len(classes)):
    raise ValueError('Shape điểm quyết định không khớp sáu lớp')
ranks = np.argsort(decision, axis=1)
top = ranks[:, -1]
second = ranks[:, -2]
if not np.array_equal(classes[top], y_pred):
    raise ValueError('Nhãn dự đoán khác điểm quyết định cao nhất')
analysis = predictions[['relative_path', 'y_true_vnd', 'y_pred_vnd']].copy()
analysis['test_index'] = np.arange(len(y_test))
analysis['correct'] = y_test == y_pred
analysis['gap_vnd'] = np.abs(y_test - y_pred)
analysis['top_decision_score'] = decision[np.arange(len(y_test)), top]
analysis['second_decision_score'] = decision[np.arange(len(y_test)), second]
analysis['score_gap'] = analysis.top_decision_score - analysis.second_decision_score
true_col = np.searchsorted(classes, y_test)
analysis['true_class_score'] = decision[np.arange(len(y_test)), true_col]
for j, label in enumerate(classes):
    analysis[f'score_{label}'] = decision[:, j]
analysis.to_csv(OUTPUT / 'all_test_decisions.csv', index=False)
analysis.loc[~analysis.correct].to_csv(OUTPUT / 'all_errors.csv', index=False)
print('Đã đối chiếu điểm quyết định của', len(analysis), 'ảnh test')


## 4. Chọn ví dụ theo quy tắc cố định

Các cặp nhầm phổ biến và cặp đóng góp chênh lệch lớn được ưu tiên. Một lỗi có khoảng cách điểm nhỏ và hai ảnh đúng được chọn làm đối chiếu. Các ví dụ có chủ đích không đại diện cho tỷ lệ nguyên nhân lỗi toàn test.


In [ ]:
selected = []
reasons = []
def choose(pool, ascending, reason):
    available = pool.loc[~pool.index.isin(selected)].sort_values(
        ['score_gap', 'relative_path'], ascending=[ascending, True])
    if len(available):
        selected.append(int(available.index[0]))
        reasons.append(reason)
errors = analysis.loc[~analysis.correct]
for row in pairs.head(3).itertuples():
    for a, b in [(row.class_a, row.class_b), (row.class_b, row.class_a)]:
        choose(errors.loc[(errors.y_true_vnd == a) & (errors.y_pred_vnd == b)],
               False, f'frequent_pair_{a}_to_{b}')
for row in directed.head(3).itertuples():
    choose(errors.loc[(errors.y_true_vnd == row.true_vnd) &
                      (errors.y_pred_vnd == row.predicted_vnd)],
           False, f'large_total_gap_{row.true_vnd}_to_{row.predicted_vnd}')
choose(errors, True, 'error_small_score_gap')
correct = analysis.loc[analysis.correct]
choose(correct, True, 'correct_small_score_gap')
choose(correct, False, 'correct_large_score_gap')
cases = analysis.loc[selected].copy().reset_index(drop=True)
cases.insert(0, 'case_id', [f'C{i+1:02d}' for i in range(len(cases))])
cases['selection_reason'] = reasons
cases.to_csv(OUTPUT / 'selected_cases.csv', index=False)
display(cases[['case_id', 'selection_reason', 'y_true_vnd',
               'y_pred_vnd', 'score_gap']])


## 5. Xem ảnh và điểm theo sáu lớp

Biểu đồ điểm hỗ trợ truy vết nhãn SVM chọn. Nó không phân tách ảnh hưởng riêng của nền, ánh sáng, góc chụp hay độ mờ; các nhận xét thị giác cần ghi sau khi xem hình.


In [ ]:
for start in range(0, len(cases), 4):
    page = cases.iloc[start:start + 4]
    fig, axes = plt.subplots(len(page), 2, figsize=(12, 3.2 * len(page)), squeeze=False)
    for r, row in enumerate(page.itertuples()):
        axes[r, 0].imshow(read_rgb224(DATA_ROOT / row.relative_path))
        axes[r, 0].set_title(f'{row.case_id}: thật {row.y_true_vnd:,} / dự đoán {row.y_pred_vnd:,}')
        axes[r, 0].axis('off')
        values = [getattr(row, f'score_{label}') for label in classes]
        axes[r, 1].bar([f'{label//1000}k' for label in classes], values)
        axes[r, 1].set_ylabel('Decision score (chưa hiệu chỉnh)')
        axes[r, 1].set_title(f'Khoảng cách hai điểm cao nhất: {row.score_gap:.4f}')
    plt.tight_layout()
    fig.savefig(OUTPUT / f'cases_page_{start//4+1:02d}.png', dpi=200)
    plt.show()
    plt.close(fig)


## 6. Ghi chú và giới hạn diễn giải

Ghi riêng điều quan sát được trên ảnh, giả thuyết nguyên nhân và điều chưa kiểm chứng. Không dùng lỗi test để đổi kernel, C, gamma hoặc feature của lượt thực nghiệm này.


In [ ]:
notes_file = OUTPUT / 'visual_review_notes.csv'
if not notes_file.exists():
    notes = cases[['case_id', 'relative_path', 'selection_reason']].copy()
    notes['observation'] = ''
    notes['hypothesis'] = ''
    notes['not_verified'] = ''
    notes.to_csv(notes_file, index=False)
save_json(OUTPUT / 'analysis_record.json', {
    'feature_id': lock['feature_id'], 'svm': lock['svm'],
    'n_test': len(analysis), 'n_errors': int((~analysis.correct).sum()),
    'n_selected_cases': len(cases),
    'selection': 'top 3 frequent pairs both ways; top 3 cost directions; one small score gap error; two correct controls',
    'decision_scores_are_probabilities': False, 'visual_causes_verified': False,
    'test_used_for_retuning': False,
    'sha256': {'lock': sha256(lock_path),
               'test_predictions': sha256(TEST / 'predictions_test.csv'),
               'test_feature': sha256(TEST / 'X_test.npy')},
})
print('Đầu ra:', OUTPUT)
